In [ ]:
# Cell 1: Secrets
import os
os.environ["HF_TOKEN"] = "..."
os.environ["COMET_API_KEY"] = "..."
os.environ["HF_MODEL_REPO"] = "sk3feel/docvqa-privacy-checkpoints"


In [ ]:
# Cell 2: Setup
!git clone https://github.com/sk3feel/hidden-data-reproduction-multimodal.git /content/course_work2026 2>/dev/null || (cd /content/course_work2026 && git pull)
%cd /content/course_work2026
!pip install -q -r requirements.txt
!pip install -q transformers accelerate Pillow pandas tqdm einops timm comet_ml huggingface_hub
!pip install -q peft bitsandbytes trl qwen-vl-utils
!python src/download_from_hf.py --repo-id sk3feel/docvqa-privacy-data

from huggingface_hub import snapshot_download
snapshot_download("sk3feel/docvqa-privacy-checkpoints", local_dir="checkpoints/", repo_type="model", token=os.environ["HF_TOKEN"])


In [ ]:
# Cell 3: Utilities and config
import gc, json, os, re, torch, pandas as pd
from pathlib import Path
from collections import defaultdict
from PIL import Image, ImageFile
from transformers import AutoModelForCausalLM, AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
from tqdm.auto import tqdm
from qwen_vl_utils import process_vision_info
import comet_ml
import matplotlib.pyplot as plt
import seaborn as sns

from src.docqa_metrics import best_metric_over_answers, build_answer_pool, estimate_random_baseline
from src.inference_scenarios import generate_scenario_image, generate_scenario_ocr

ImageFile.LOAD_TRUNCATED_IMAGES = True
MAX_IMAGE_SIDE = 1024
EXTRACTION_DIR = Path("artifacts/privacy_attacks/extraction")
EXTRACTION_DIR.mkdir(parents=True, exist_ok=True)
SUCCESS_PREVIEW_DIR = EXTRACTION_DIR / "success_previews"
SUCCESS_PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

FLORENCE_SCENARIOS = ["original", "img_black", "img_blur_20", "img_blur_50"]
QWEN_IMAGE_ONLY_SCENARIOS = ["original", "img_black", "img_blur_20", "img_blur_50"]
QWEN_IMAGE_OCR_SCENARIOS = ["img_black__ocr_mask__k_0", "img_none__ocr_mask__k_0"]
QWEN_OCR_SCENARIO_MAP = {
    "img_black__ocr_mask__k_0": "ocr_mask__img_black__k_0",
    "img_none__ocr_mask__k_0": "ocr_mask__img_none__k_0",
}


def save_csv_with_comet(df, path, experiment, table_name=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    csv_string = df.to_csv(index=False)
    experiment.log_asset(str(path), file_name=path.name)
    experiment.log_asset_data(csv_string, file_name=path.name)
    if table_name is not None:
        experiment.log_table(table_name, df)
    return path


def safe_open_image(path, max_side=MAX_IMAGE_SIDE):
    image = Image.open(path).convert("RGB")
    if max(image.size) > max_side:
        image.thumbnail((max_side, max_side), Image.LANCZOS)
    return image


def resize_generated_image(image, max_side=MAX_IMAGE_SIDE):
    image = image.convert("RGB")
    if max(image.size) > max_side:
        image.thumbnail((max_side, max_side), Image.LANCZOS)
    return image


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def enrich_with_field_types(records, manifest_path):
    manifest = []
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                manifest.append(json.loads(line))
    type_map = {}
    for row in manifest:
        eid = str(row.get("example_id", row.get("question_id", "")))
        type_map[eid] = row.get("coarse_field_type", "OTHER")
    for rec in records:
        rec["coarse_field_type"] = type_map.get(rec["example_id"], "OTHER")
    return records


def stratified_sample(records, n, type_key="coarse_field_type"):
    by_type = defaultdict(list)
    for rec in records:
        by_type[rec[type_key]].append(rec)
    types = sorted(by_type.keys())
    per_type = n // len(types)
    result = []
    for ft in types:
        result.extend(by_type[ft][:per_type])
    used = {r["example_id"] for r in result}
    for rec in records:
        if len(result) >= n:
            break
        if rec["example_id"] not in used:
            result.append(rec)
    return result[:n]


def normalize_text(text):
    text = str(text or "")
    text = re.sub(r"<[^>]+>", " ", text)
    text = " ".join(text.replace("\n", " ").split()).strip()
    return text


def normalize_em(text):
    text = str(text or "").lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def qwen_user_text(question, ocr_text=None):
    if ocr_text is None:
        return f"Answer the question about this document image. Give only the answer value, nothing else.\nQuestion: {question}"
    return (
        "Answer the question about this document image using the OCR text provided. "
        "Give only the answer value, nothing else.\n"
        f"OCR: {ocr_text}\nQuestion: {question}"
    )


def load_florence(checkpoint_path=None):
    model_name = str(checkpoint_path) if checkpoint_path is not None else "microsoft/Florence-2-base"
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).to("cuda")
    model.eval()
    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    return model, processor


def load_qwen(base_model_name, adapter_path=None):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
    base = Qwen2VLForConditionalGeneration.from_pretrained(base_model_name, quantization_config=bnb, device_map="auto", trust_remote_code=True)
    if adapter_path is not None:
        model = PeftModel.from_pretrained(base, str(adapter_path))
    else:
        model = base
    model.eval()
    processor = AutoProcessor.from_pretrained(base_model_name, trust_remote_code=True)
    return model, processor


def generate_prediction_florence(model, processor, image, question):
    inputs = processor(text=f"<VQA>{question}", images=image, return_tensors="pt").to(model.device)
    generate_inputs = {"input_ids": inputs["input_ids"], "pixel_values": inputs["pixel_values"]}
    with torch.no_grad():
        gen = model.generate(**generate_inputs, max_new_tokens=50)
    return normalize_text(processor.batch_decode(gen, skip_special_tokens=True)[0])


def generate_prediction_qwen(model, processor, image, question, ocr_text=None):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": qwen_user_text(question, ocr_text=ocr_text)},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    img_inputs, _ = process_vision_info(messages)
    img_input = img_inputs[0] if isinstance(img_inputs, list) else img_inputs
    inputs = processor(text=[text], images=[img_input], return_tensors="pt", padding=True)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=50)
    pred = processor.batch_decode(gen[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
    return normalize_text(pred)


def run_extraction(model, processor, records, scenarios, model_kind, mode):
    rows = []
    for rec in tqdm(records, desc=f"extraction:{model_kind}:{mode}"):
        for scenario in scenarios:
            internal_scenario = QWEN_OCR_SCENARIO_MAP.get(scenario, scenario)
            image = resize_generated_image(generate_scenario_image(rec, internal_scenario))
            ocr_text = generate_scenario_ocr(rec, internal_scenario) if mode == "image_ocr" else None

            if model_kind == "florence":
                prediction = generate_prediction_florence(model, processor, image, rec["question"])
            else:
                prediction = generate_prediction_qwen(model, processor, image, rec["question"], ocr_text=ocr_text)

            rows.append({
                "example_id": rec["example_id"],
                "split": rec["split"],
                "scenario": scenario,
                "prediction": prediction,
                "question": rec["question"],
                "answer": rec["answer"],
                "answers": list(rec.get("answers") or [rec["answer"]]),
                "coarse_type": rec["coarse_field_type"],
                "ocr_text_used": ocr_text or "",
                "ocr_source": "generate_scenario_ocr" if mode == "image_ocr" else "",
            })
    return pd.DataFrame(rows)


def score_predictions(predictions_df, answer_pools):
    rows = []
    for row in predictions_df.to_dict(orient="records"):
        answers = row["answers"]
        random_record = {
            "example_id": row["example_id"],
            "coarse_field_type": row["coarse_type"],
            "answers": answers,
            "answer": row["answer"],
            "local_row_id": 0,
        }
        exact_match, token_f1 = best_metric_over_answers(row["prediction"], answers)
        random_em, random_f1 = estimate_random_baseline(random_record, answer_pools[row["split"]])
        rows.append({
            **row,
            "exact_match": exact_match,
            "token_f1": token_f1,
            "random_em": random_em,
            "random_f1": random_f1,
            "corrected_em": exact_match - random_em,
            "corrected_f1": token_f1 - random_f1,
            "normalized_prediction": normalize_em(row["prediction"]),
            "normalized_answer": normalize_em(row["answer"]),
        })
    return pd.DataFrame(rows)


def compute_metrics(scored_df):
    rows = []
    for (split, scenario, coarse_type), group in scored_df.groupby(["split", "scenario", "coarse_type"]):
        rows.append({
            "split": split,
            "scenario": scenario,
            "coarse_type": coarse_type,
            "n_examples": len(group),
            "exact_match": float(group["exact_match"].mean()),
            "token_f1": float(group["token_f1"].mean()),
            "random_em": float(group["random_em"].mean()),
            "random_f1": float(group["random_f1"].mean()),
            "corrected_em": float(group["corrected_em"].mean()),
            "corrected_f1": float(group["corrected_f1"].mean()),
        })
    overall_rows = []
    for (split, scenario), group in scored_df.groupby(["split", "scenario"]):
        overall_rows.append({
            "split": split,
            "scenario": scenario,
            "coarse_type": "ALL",
            "n_examples": len(group),
            "exact_match": float(group["exact_match"].mean()),
            "token_f1": float(group["token_f1"].mean()),
            "random_em": float(group["random_em"].mean()),
            "random_f1": float(group["random_f1"].mean()),
            "corrected_em": float(group["corrected_em"].mean()),
            "corrected_f1": float(group["corrected_f1"].mean()),
        })
    return pd.DataFrame(rows + overall_rows).sort_values(["split", "scenario", "coarse_type"]).reset_index(drop=True)


def run_model_extraction_experiment(model, processor, model_tag, model_kind, mode, scenarios, seen_records, unseen_records, answer_pools, experiment_name):
    experiment = comet_ml.Experiment(api_key=os.environ["COMET_API_KEY"], workspace="scfeel", project_name="qwen3-1")
    experiment.set_name(experiment_name)
    experiment.log_parameters({
        "model_tag": model_tag,
        "mode": mode,
        "scenarios": scenarios,
        "n_seen": len(seen_records),
        "n_unseen": len(unseen_records),
    })

    combined_records = seen_records + unseen_records
    record_lookup = {(str(rec["example_id"]), rec["split"]): rec for rec in combined_records}
    predictions_df = run_extraction(model, processor, combined_records, scenarios, model_kind=model_kind, mode=mode)
    scored_df = score_predictions(predictions_df, answer_pools)
    metrics_df = compute_metrics(scored_df)
    success_df = scored_df[(scored_df["split"] == "seen") & (scored_df["scenario"] != "original") & (scored_df["exact_match"] >= 1.0)].head(20).copy()

    prefix = f"{model_tag}_{mode}"
    predictions_path = EXTRACTION_DIR / f"{prefix}_predictions.csv"
    scored_path = EXTRACTION_DIR / f"{prefix}_scored.csv"
    metrics_path = EXTRACTION_DIR / f"{prefix}_metrics.csv"
    success_path = EXTRACTION_DIR / f"{prefix}_successful_examples.csv"
    save_csv_with_comet(predictions_df, predictions_path, experiment)
    save_csv_with_comet(scored_df, scored_path, experiment)
    save_csv_with_comet(metrics_df, metrics_path, experiment, table_name=metrics_path.name)
    save_csv_with_comet(success_df, success_path, experiment, table_name=success_path.name)

    for _, row in metrics_df[metrics_df["coarse_type"] == "ALL"].iterrows():
        metric_name = f"{row['split']}_{row['scenario']}_corrected_em".replace("+", "__").replace("-", "_")
        experiment.log_metric(metric_name, float(row["corrected_em"]))

    if not success_df.empty:
        preview_dir = SUCCESS_PREVIEW_DIR / prefix
        preview_dir.mkdir(parents=True, exist_ok=True)
        for idx, row in success_df.head(5).reset_index(drop=True).iterrows():
            record = record_lookup.get((str(row["example_id"]), row["split"]))
            if record is None:
                continue
            internal_scenario = QWEN_OCR_SCENARIO_MAP.get(row["scenario"], row["scenario"])
            preview_image = resize_generated_image(generate_scenario_image(record, internal_scenario))
            preview_path = preview_dir / f"{idx}_{row['example_id']}_{row['scenario']}.png"
            preview_image.save(preview_path)
            experiment.log_image(str(preview_path), name=preview_path.name)

    experiment.end()
    return predictions_df, scored_df, metrics_df, success_df


def cleanup_gpu():
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# Cell 4: Data preparation
qwen_train_ids_path = Path("artifacts/finetuning_generative/qwen_train_200_ids.csv")
assert qwen_train_ids_path.exists(), "Missing qwen_train_200_ids.csv from notebook 21"
qwen_train_ids = pd.read_csv(qwen_train_ids_path)["example_id"].astype(str).tolist()

train_manifest = load_jsonl("artifacts/docqa_recovery/benchmark_train/manifest.jsonl")
validation_manifest = load_jsonl("artifacts/docqa_recovery/benchmark/manifest.jsonl")

def manifest_to_record(row, split, image_root):
    image_name = str(row.get("image_path", "")).replace("\\", "/").split("/")[-1]
    answer_value = row["answers"][0] if isinstance(row.get("answers"), list) and row.get("answers") else row.get("answer", "")
    record = dict(row)
    record["example_id"] = str(row.get("example_id", row.get("question_id", "")))
    record["split"] = split
    record["image_path"] = str(Path(image_root) / image_name)
    record["answer"] = answer_value
    record["coarse_field_type"] = row.get("coarse_field_type", "OTHER")
    return record


train_records_full = [
    manifest_to_record(r, "seen", "artifacts/docqa_recovery/benchmark_train/images/original")
    for r in train_manifest
]
validation_records_full = [
    manifest_to_record(r, "unseen", "artifacts/docqa_recovery/benchmark/images/original")
    for r in validation_manifest
]

qwen_seen = [r for r in train_records_full if r["example_id"] in set(qwen_train_ids)]
florence_seen = stratified_sample(train_records_full, 200)
unseen_200 = stratified_sample(validation_records_full, 200)

florence_answer_pools = {
    "seen": build_answer_pool(florence_seen),
    "unseen": build_answer_pool(unseen_200),
}
qwen_answer_pools = {
    "seen": build_answer_pool(qwen_seen),
    "unseen": build_answer_pool(unseen_200),
}

print({
    "florence_seen": len(florence_seen),
    "qwen_seen": len(qwen_seen),
    "unseen_200": len(unseen_200),
})


In [ ]:
# Cell 5: Baseline Florence extraction
model, processor = load_florence()
baseline_outputs = run_model_extraction_experiment(
    model=model,
    processor=processor,
    model_tag="baseline_florence2",
    model_kind="florence",
    mode="image_only",
    scenarios=FLORENCE_SCENARIOS,
    seen_records=florence_seen,
    unseen_records=unseen_200,
    answer_pools=florence_answer_pools,
    experiment_name="extraction-baseline-florence2",
)
display(baseline_outputs[2])
del model, processor
cleanup_gpu()


In [ ]:
# Cell 6: Florence-2 epoch_30 extraction
model, processor = load_florence("checkpoints/florence2/epoch_30")
florence_outputs = run_model_extraction_experiment(
    model=model,
    processor=processor,
    model_tag="florence2_epoch30",
    model_kind="florence",
    mode="image_only",
    scenarios=FLORENCE_SCENARIOS,
    seen_records=florence_seen,
    unseen_records=unseen_200,
    answer_pools=florence_answer_pools,
    experiment_name="extraction-florence2-epoch30",
)
display(florence_outputs[2])
del model, processor
cleanup_gpu()


In [ ]:
# Cell 7: Qwen2-VL-2B extraction
model, processor = load_qwen("Qwen/Qwen2-VL-2B-Instruct", "checkpoints/qwen2b/epoch_5")
qwen2b_image_only_outputs = run_model_extraction_experiment(
    model=model,
    processor=processor,
    model_tag="qwen2b_epoch5",
    model_kind="qwen",
    mode="image_only",
    scenarios=QWEN_IMAGE_ONLY_SCENARIOS,
    seen_records=qwen_seen,
    unseen_records=unseen_200,
    answer_pools=qwen_answer_pools,
    experiment_name="extraction-qwen2b-epoch5-imageonly",
)
display(qwen2b_image_only_outputs[2])
del model, processor
cleanup_gpu()

model, processor = load_qwen("Qwen/Qwen2-VL-2B-Instruct", "checkpoints/qwen2b/epoch_5")
qwen2b_image_ocr_outputs = run_model_extraction_experiment(
    model=model,
    processor=processor,
    model_tag="qwen2b_epoch5",
    model_kind="qwen",
    mode="image_ocr",
    scenarios=QWEN_IMAGE_OCR_SCENARIOS,
    seen_records=qwen_seen,
    unseen_records=unseen_200,
    answer_pools=qwen_answer_pools,
    experiment_name="extraction-qwen2b-epoch5-imageocr",
)
display(qwen2b_image_ocr_outputs[2])
del model, processor
cleanup_gpu()


In [ ]:
# Cell 8: Qwen2-VL-7B extraction
model, processor = load_qwen("Qwen/Qwen2-VL-7B-Instruct", "checkpoints/qwen7b/epoch_5")
qwen7b_image_only_outputs = run_model_extraction_experiment(
    model=model,
    processor=processor,
    model_tag="qwen7b_epoch5",
    model_kind="qwen",
    mode="image_only",
    scenarios=QWEN_IMAGE_ONLY_SCENARIOS,
    seen_records=qwen_seen,
    unseen_records=unseen_200,
    answer_pools=qwen_answer_pools,
    experiment_name="extraction-qwen7b-epoch5-imageonly",
)
display(qwen7b_image_only_outputs[2])
del model, processor
cleanup_gpu()


In [ ]:
# Cell 9: Model summaries
summary_rows = []
for name, outputs in {
    "baseline_florence2": baseline_outputs,
    "florence2_epoch30": florence_outputs,
    "qwen2b_epoch5_image_only": qwen2b_image_only_outputs,
    "qwen2b_epoch5_image_ocr": qwen2b_image_ocr_outputs,
    "qwen7b_epoch5_image_only": qwen7b_image_only_outputs,
}.items():
    metrics_df = outputs[2]
    overall = metrics_df[metrics_df["coarse_type"] == "ALL"].copy()
    overall["model"] = name
    summary_rows.append(overall)

summary_df = pd.concat(summary_rows, ignore_index=True)
summary_path = EXTRACTION_DIR / "extraction_summary.csv"

summary_experiment = comet_ml.Experiment(api_key=os.environ["COMET_API_KEY"], workspace="scfeel", project_name="qwen3-1")
summary_experiment.set_name("extraction-summary")
save_csv_with_comet(summary_df, summary_path, summary_experiment, table_name="extraction_summary.csv")

image_only_df = summary_df[(summary_df["model"].isin(["baseline_florence2", "florence2_epoch30", "qwen2b_epoch5_image_only", "qwen7b_epoch5_image_only"])) & (summary_df["split"] == "seen")].copy()
image_only_df["model_label"] = image_only_df["model"].map({
    "baseline_florence2": "Florence-2 base",
    "florence2_epoch30": "Florence-2 (????? 30)",
    "qwen2b_epoch5_image_only": "Qwen2-VL-2B (????? 5)",
    "qwen7b_epoch5_image_only": "Qwen2-VL-7B (????? 5)",
})
heatmap_df = image_only_df.pivot(index="model_label", columns="scenario", values="corrected_em")
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax)
ax.set_title("Extraction ?? image-only ????????? (seen)")
ax.set_xlabel("????????")
ax.set_ylabel("??????")
plt.tight_layout()
summary_experiment.log_figure("extraction_heatmap_image_only_seen", fig)
plt.close(fig)

image_ocr_df = summary_df[(summary_df["model"] == "qwen2b_epoch5_image_ocr") & (summary_df["split"] == "seen")].copy()
if not image_ocr_df.empty:
    heatmap_df = image_ocr_df.assign(model_label="Qwen2-VL-2B (epoch 5, image+OCR)").pivot(index="model_label", columns="scenario", values="corrected_em")
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="YlOrBr", ax=ax)
    ax.set_title("Extraction ?? image+OCR ????????? (seen)")
    ax.set_xlabel("????????")
    ax.set_ylabel("")
    plt.tight_layout()
    summary_experiment.log_figure("extraction_heatmap_image_ocr_seen", fig)
    plt.close(fig)

display(summary_df)
summary_experiment.end()


In [ ]:
# Cell 10: Saved outputs
saved_csvs = sorted(str(path) for path in EXTRACTION_DIR.glob("*.csv"))
display(pd.DataFrame({"csv_path": saved_csvs}))
print({"csv_count": len(saved_csvs), "output_dir": str(EXTRACTION_DIR)})
